In [258]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import TFBertModel
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Flatten, Layer, MultiHeadAttention
from tensorflow.keras.models import Sequential
from lime.lime_tabular import LimeTabularExplainer
from lime.lime_text import LimeTextExplainer
import shap
from collections import defaultdict

In [6]:

# Load and preprocess data
cali_housing_path = '../data/California_Houses.csv'
RANDOM_SEED = 492
cali_df = pd.read_csv(cali_housing_path)
y = cali_df['Median_House_Value']
X = cali_df.drop(columns=['Median_House_Value'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [271]:
class FeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, embed_dim):
        super(FeatureTokenizer, self).__init__()
        self.num_features = num_features
        self.embed_dim = embed_dim
        
        self.feature_weights = self.add_weight(shape=(num_features, embed_dim),
                                               initializer='random_normal',
                                               trainable=True)
        self.feature_bias = self.add_weight(shape=(num_features, embed_dim),
                                            initializer='random_normal',
                                            trainable=True)
        self.layer_norm = LayerNormalization()
    
    def call(self, inputs):
        embeddings = inputs[..., tf.newaxis] * self.feature_weights + self.feature_bias
        embeddings = self.layer_norm(embeddings)
        return embeddings

# Tokenizer Example Usage
num_features = X_train.shape[1]
embed_dim = 20
feature_tokenizer = FeatureTokenizer(num_features, embed_dim)

# Tokenize and encode inputs
train_encodings = feature_tokenizer(X_train_scaled)
test_encodings = feature_tokenizer(X_test_scaled)

In [407]:
# Define TransformerBlock
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, attention_mask=None, training=False):
        attn_output = self.att(inputs, inputs, attention_mask=attention_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Define FTTransformer
class FTTransformer(tf.keras.Model):
    def __init__(self, dim, dim_out, depth, heads, ff_dim, mlp_hidden, attn_dropout=0.1, ff_dropout=0.1):
        super(FTTransformer, self).__init__()
        self.transformers = [TransformerBlock(dim, heads, ff_dim, rate=attn_dropout) for _ in range(depth)]
        self.flatten_transformer_output = tf.keras.layers.Flatten()
        self.mlp_layers = [Dense(size, activation=activation) for size, activation in mlp_hidden]
        self.output_layer = Dense(dim_out)
    
    def call(self, inputs, attention_mask=None, training=False):
        x = inputs
        for transformer in self.transformers:
            x = transformer(x, attention_mask=attention_mask, training=training)
        x = self.flatten_transformer_output(x)
        for mlp_layer in self.mlp_layers:
            x = mlp_layer(x)
        return self.output_layer(x)


In [300]:
class CrossTransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.rate = rate
        
        # Self-attention layers
        self.attention_self = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        
        # Cross-attention layers (additional for cross-attention)
        self.attention_cross = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        
        # Feed-forward network
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        
        # Layer normalizations and dropout
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)
        
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)
        self.dropout3 = Dropout(rate)

    def call(self, inputs, encoder_output=None, self_attention_mask=None, cross_attention_mask=None, training=False):
        # Self-attention
        attn_output_self = self.attention_self(inputs, inputs, attention_mask=self_attention_mask)
        attn_output_self = self.dropout1(attn_output_self, training=training)
        out1 = self.layernorm1(inputs + attn_output_self)
        
        # Cross-attention (if encoder_output is provided)
        if encoder_output is not None:
            attn_output_cross = self.attention_cross(out1, encoder_output, attention_mask=cross_attention_mask)
            attn_output_cross = self.dropout2(attn_output_cross, training=training)
            out2 = self.layernorm2(out1 + attn_output_cross)
        else:
            out2 = out1
        
        # Feed-forward network
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=training)
        out3 = self.layernorm3(out2 + ffn_output)
        
        return out3
    
    
class CrossFTTransformer(tf.keras.Model):
    def __init__(self, dim, dim_out, depth, heads, ff_dim, mlp_hidden, attn_dropout=0.1, ff_dropout=0.1):
        super(FTTransformer, self).__init__()
        self.transformers = [TransformerBlock(dim, heads, ff_dim, rate=attn_dropout) for _ in range(depth)]
        self.flatten_transformer_output = tf.keras.layers.Flatten()
        self.mlp_layers = [Dense(size, activation=activation) for size, activation in mlp_hidden]
        self.output_layer = Dense(dim_out)
    
    def call(self, inputs, encoder_output=None, self_attention_mask=None, cross_attention_mask=None, training=False):
        x = inputs
        for transformer in self.transformers:
            x = transformer(x, encoder_output=encoder_output, self_attention_mask=self_attention_mask, 
                            cross_attention_mask=cross_attention_mask, training=training)
        x = self.flatten_transformer_output(x)
        for mlp_layer in self.mlp_layers:
            x = mlp_layer(x)
        return self.output_layer(x)


In [420]:

class BERTMLP(tf.keras.Model):
    def __init__(self, dim=20, dim_out=1, depth=2, heads=4, ff_dim=128, mlp_hidden=[(128, 'relu'), (64, 'relu')], hidden_dim=64, dropout_rate=0.1):
        super(BERTMLP, self).__init__()
        self.transformer = FTTransformer(dim, dim_out, depth, heads, ff_dim, mlp_hidden)
        self.dense1 = Dense(hidden_dim, activation='relu')
        self.dropout1 = Dropout(dropout_rate)
        self.dense2 = Dense(hidden_dim, activation='relu')
        self.dropout2 = Dropout(dropout_rate)
        self.dense3 = Dense(1, activation='linear')
        self.heads = heads

    def call(self, inputs, attention_mask=None, training=False):
        batch_size = tf.shape(inputs)[0]
        seq_length = tf.shape(inputs)[1]
        if attention_mask is None:
            attention_mask = tf.ones((batch_size, seq_length, seq_length), dtype=tf.float32)
        bert_output = self.transformer(inputs, attention_mask=attention_mask, training=training)
        x = self.dense1(bert_output)
        x = self.dropout1(x, training=training)
        x = self.dense2(x)
        x = self.dropout2(x, training=training)
        x = self.dense3(x)
        return x


def aggregate_embeddings(embeddings, method='mean'):
    if method == 'mean':
        return np.mean(embeddings, axis=-1)
    elif method == 'max':
        return np.max(embeddings, axis=-1)
    # Add more methods as needed

def explain_lime_custom(model, instance, train_data, num_features, embed_dim):
    # Aggregate the embeddings for each feature in the training data
    train_data_agg = aggregate_embeddings(train_data, method='mean')
    
    # Generate feature names based on the number of features
    feature_names = [f'feature_{i}' for i in range(num_features)]
    
    explainer = LimeTabularExplainer(
        training_data=train_data_agg,
        feature_names=feature_names,
        mode='regression'
    )
    
    # Aggregate the embeddings for the instance
    instance_agg = aggregate_embeddings(instance, method='mean')
    
    # Create a wrapper for the model's predict function
    # def predict_wrapper(x):
    #     x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
    #     x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
    #     return model(x_3d, training=True).numpy()
    
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    
    explanation = explainer.explain_instance(
        instance_agg[0], 
        predict_wrapper, 
        num_features=num_features
    )
    
    return explanation

def explain_lime_dense_layers(model, instance, num_features, embed_dim):
    # Generate feature names based on the number of features
    feature_names = [f'feature_{i}' for i in range(num_features)]
    # Create a dummy dataset with one sample for initialization
    dummy_data = np.zeros((1, num_features))
    
    explainer = LimeTabularExplainer(
        training_data=dummy_data,  # No need to pass training data for dense layer explanation
        feature_names=feature_names,
        mode='regression'
    )
    
    def predict_wrapper(x):
        # Simulate the output of dense layers only
        x_3d = tf.convert_to_tensor(x, dtype=tf.float32)
        x = model.dense1(x_3d)
        x = model.dropout1(x, training=True)
        x = model.dense2(x)
        x = model.dropout2(x, training=True)
        x = model.dense3(x)
        return x.numpy()  # Return numpy array since LIME expects numpy predictions
    
    instance_agg = aggregate_embeddings(instance, method='mean')
    
    explanation = explainer.explain_instance(
        instance_agg[0],
        predict_wrapper,
        num_features=num_features
    )
    
    return explanation

def explain_shap_custom(model, instance, train_data, num_features, embed_dim):
    # Aggregate the embeddings for each feature in the training data
    train_data_agg = aggregate_embeddings(train_data.numpy(), method='mean')
    
    # Create a wrapper for the model's predict function
    # def predict_wrapper(x):
    #     x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
    #     x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
    #     return model(x_3d, training=True).numpy()
    
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    
    explainer = shap.KernelExplainer(predict_wrapper, train_data_agg)
    instance_agg = aggregate_embeddings(instance.numpy().reshape(1, num_features, embed_dim), method='mean')
    shap_values = explainer.shap_values(instance_agg)
    return shap_values

def predict_with_explanation(model, inputs, num_samples=10):
    num_features = inputs.shape[1]
    embed_dim = inputs.shape[2]
    
    predictions = []
    lime_explanations = []
    shap_explanations = []
    
    for i in range(len(inputs)):
        instance = inputs[i:i+1]
        for _ in range(num_samples):
            prediction = model(inputs[i:i+1], training=True)
            predictions.append(prediction)
            
            lime_explanation = explain_lime_dense_layers(model, instance, num_features, embed_dim)
            # shap_explanation = explain_shap_custom(model, instance, inputs, num_features, embed_dim)
            
            lime_explanations.append(lime_explanation)
            # shap_explanations.append(shap_explanation)
            
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy(), lime_explanations, shap_explanations


def predict_with_uncertainty(model, inputs, num_samples=10):
    predictions = []
    for _ in range(num_samples):
        prediction = model(inputs, training=True)
        predictions.append(prediction)
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy()

class ModelWithUncertainty(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, method='mc_dropout'):
        super(ModelWithUncertainty, self).__init__()
        self.method = method
        self.model = BERTMLP(dim=input_dim, hidden_dim=hidden_dim, dropout_rate=dropout_rate)
    
    def call(self, inputs, training=False):
        return self.model(inputs, training=training)
    
    def predict_with_explanation(self, inputs, num_samples):
        return predict_with_explanation(self.model, inputs, num_samples)
    
    def predict_with_uncertainty(self, inputs, num_samples):
        return predict_with_uncertainty(self.model, inputs, num_samples)
    

In [382]:
class MLP(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1):
        super(MLP, self).__init__()
        self.dense1 = Dense(hidden_dim, activation='relu')
        self.dropout1 = Dropout(dropout_rate)
        self.dense2 = Dense(hidden_dim, activation='relu')
        self.dropout2 = Dropout(dropout_rate)
        self.dense3 = Dense(1, activation='linear')

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.dropout1(x, training=training)
        x = self.dense2(x)
        x = self.dropout2(x, training=training)
        x = self.dense3(x)
        return x

# Helper function to aggregate embeddings
def aggregate_embeddings(data, method='mean'):
    if method == 'mean':
        return np.mean(data, axis=-1)
    else:
        raise ValueError("Unsupported aggregation method")


# # Define LIME and SHAP explanation functions
# def mlp_explain_lime_custom(model, instance, train_data, num_features, embed_dim):
#     # Aggregate the embeddings for each feature in the training data
#     train_data_agg = aggregate_embeddings(train_data, method='mean')
#     print(f"Aggregated training data shape: {train_data_agg.shape}")
#     # Generate feature names based on the number of features
#     feature_names = [f'feature_{i}' for i in range(num_features)]
#     
#     explainer = LimeTabularExplainer(
#         training_data=train_data_agg,
#         feature_names=feature_names,
#         mode='regression'
#     )
#     
#     # Aggregate the embeddings for the instance
#     instance_agg = aggregate_embeddings(instance, method='mean').flatten()
#     print(f"Aggregated instance shape: {instance_agg.shape}")
#     # Create a wrapper for the model's predict function
#     def predict_wrapper(x):
#         x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
#         x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
#         return model(x_3d, training=False).numpy().flatten()
#     
#     explanation = explainer.explain_instance(
#         instance_agg, 
#         predict_wrapper, 
#         num_features=num_features
#     )
#     
#     return explanation
# 
# def mlp_explain_shap_custom(model, instance, train_data, num_features, embed_dim):
#     # Aggregate the embeddings for each feature in the training data
#     train_data_agg = aggregate_embeddings(train_data, method='mean')
#     
#     # Create a wrapper for the model's predict function
#     def predict_wrapper(x):
#         x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
#         x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
#         return model(x_3d, training=False).numpy().flatten()
#     
#     explainer = shap.KernelExplainer(predict_wrapper, train_data_agg)
#     instance_agg = aggregate_embeddings(instance, method='mean').flatten()
#     shap_values = explainer.shap_values(instance_agg)
#     return shap_values
   
def mlp_explain_lime_custom(model, instance, train_data, num_features, embed_dim):
    # Aggregate the embeddings for each feature in the training data
    train_data_agg = aggregate_embeddings(train_data.numpy(), method='mean')
    
    # Generate feature names based on the number of features
    feature_names = [f'feature_{i}' for i in range(num_features)]
    
    explainer = LimeTabularExplainer(
        training_data=train_data_agg,
        feature_names=feature_names,
        mode='regression'
    )
    
    # Aggregate the embeddings for the instance
    instance_agg = aggregate_embeddings(instance.numpy().reshape(1, num_features, embed_dim), method='mean')
    
    # Create a wrapper for the model's predict function
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    
    explanation = explainer.explain_instance(
        instance_agg[0], 
        predict_wrapper, 
        num_features=num_features
    )
    
    return explanation

def mlp_explain_shap_custom(model, instance, train_data, num_features, embed_dim):
    # Aggregate the embeddings for each feature in the training data
    train_data_agg = aggregate_embeddings(train_data.numpy(), method='mean')
    
    # Create a wrapper for the model's predict function
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    
    explainer = shap.KernelExplainer(predict_wrapper, train_data_agg)
    instance_agg = aggregate_embeddings(instance.numpy().reshape(1, num_features, embed_dim), method='mean')
    shap_values = explainer.shap_values(instance_agg)
    return shap_values

def mlp_predict_with_uncertainty_exp(model, inputs, train_encodings, num_samples=10):
    num_features = inputs.shape[1]
    embed_dim = inputs.shape[2]
    predictions = []
    lime_explanations = []
    shap_explanations = []
    for _ in range(num_samples):
        prediction = model(inputs, training=True)
        predictions.append(prediction)
        lime_explanation = mlp_explain_lime_custom(model, inputs, train_encodings, num_features, embed_dim)
        shap_explanation = mlp_explain_shap_custom(model, inputs, train_encodings, num_features, embed_dim)
        lime_explanations.append(lime_explanation)
        shap_explanations.append(shap_explanation)
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy(), lime_explanations, shap_explanations

def mlp_predict_uncertain(model, inputs, num_samples=10): 
    predictions = []
    for _ in range(num_samples):
        prediction = model(inputs, training=True)
        predictions.append(prediction)
    predictions = tf.stack(predictions, axis=0)
    prediction_mean = tf.reduce_mean(predictions, axis=0)
    prediction_std = tf.math.reduce_std(predictions, axis=0)
    return prediction_mean.numpy(), prediction_std.numpy()


class MLPModelWithUncertainty(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, method='mc_dropout'):
        super(MLPModelWithUncertainty, self).__init__()
        self.method = method
        if method == 'mc_dropout':
            self.model = MLP(input_dim, hidden_dim, dropout_rate)
        # elif method == 'dropconnect':
        #     self.model = MLPDropConnect(input_dim, hidden_dim, dropout_rate)
        else:
            raise ValueError("Method should be 'mc_dropout' or 'dropconnect'")
    
    def call(self, inputs, training=False):
        return self.model(inputs, training=training)
    
    def predict_with_uncertain(self, inputs, num_samples):
        return mlp_predict_uncertain(self.model, inputs, num_samples)
    
    def predict_with_exp(self, inputs, train_encodings, num_samples):
        return mlp_predict_with_uncertainty_exp(self.model, inputs, train_encodings, num_samples)

    def explain_lime(self, instance, data, num_features, embed_dim):
        return mlp_explain_lime_custom(self.model, instance, data, num_features=num_features, embed_dim=embed_dim)

    def explain_shap(self, instance, data, num_features, embed_dim):
        return mlp_explain_shap_custom(self.model, instance, data, num_features, embed_dim)




In [338]:
test_dat_agg = aggregate_embeddings(test_encodings[128:129], method='mean')

In [339]:
print(test_dat_agg)

(16512, 13)


In [335]:
print(test_encodings[128:129])

tf.Tensor(
[[[-0.6767724   0.2135692  -1.0125668   0.8075836  -0.85853475
   -0.5841223   0.19556656 -0.54865104  0.5416011   0.19216248
   -1.268037    1.8469337   1.5839678  -1.6689327  -0.16438222
    0.691104   -0.02060734 -0.0163488   0.30866903  0.4377978 ]
  [-1.2065216   0.20794004  0.06998681 -0.20940246 -2.023868
   -0.5470577   0.3790791   0.13449195 -1.3094621   0.05823144
    0.3682589  -0.26861805  0.79618263  2.351961    1.6538516
   -0.46569926 -0.39567918 -0.15154324 -0.25180215  0.80967087]
  [ 0.22668232 -1.612676    0.60676616 -0.8278076  -0.22251606
    1.6148705   1.9349447   0.11337744  0.6431177  -0.07312313
   -0.19181345  1.0651214  -0.28851968  0.05915566  0.3588814
   -0.7538369  -0.56910914 -1.5652957  -0.6133909   0.0951709 ]
  [ 0.48211765 -1.1466417  -0.510429   -0.8602455  -0.9504551
   -0.02878878  1.287363   -0.30405092  2.451831   -0.3160292
    0.3829762  -0.3376729   0.647525    0.14520551  1.2771659
   -1.6532903   0.58968556 -0.13210866 -0.631533

In [409]:
dense_size = 300
dropout_rate = 0.1
num_samples = 10
num_features = train_encodings.shape[1]  # This should be 13 based on the error message
embed_dim = train_encodings.shape[2]  # This should match the last dimension of your input

In [398]:
print(embed_dim)

20


In [421]:

# Create the model
model = ModelWithUncertainty(input_dim=embed_dim, hidden_dim=dense_size, dropout_rate=dropout_rate, method='mc_dropout')

# Compile the model
model.compile(optimizer='adam', loss='mse')

In [393]:
mlp_model = MLPModelWithUncertainty(input_dim=train_encodings.shape[-1], hidden_dim=dense_size, method='mc_dropout')
mlp_model.compile(optimizer='adam', loss='mse')

In [394]:
mlp_history = mlp_model.fit(
    train_encodings, 
    y_train, 
    epochs=2, 
    validation_split=0.2
)

Epoch 1/2
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 54227107840.0000 - val_loss: 28560564224.0000
Epoch 2/2
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 22981144576.0000 - val_loss: 14197295104.0000


In [313]:
print(train_encodings.shape)

(16512, 13, 20)


In [422]:

# Train the model
model_history = model.fit(
    train_encodings, 
    y_train, 
    epochs=2,  # You might want to increase this
    batch_size=32, 
    validation_split=0.2,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/2
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 28240998400.0000 - val_loss: 3971728128.0000 - learning_rate: 0.0010
Epoch 2/2
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 4022847488.0000 - val_loss: 3719124480.0000 - learning_rate: 0.0010


In [423]:

# Predict with uncertainty
mean_pred, std_pred = model.predict_with_uncertainty(test_encodings[128:129], num_samples=num_samples)

In [395]:
mlp_pred_mean, mlp_pred_std = mlp_model.predict_with_uncertain(test_encodings[128:129], num_samples=num_samples)

In [425]:
print(f"mean shape: {mean_pred}, uncertain shape: {std_pred}")

mean shape: [[426382.56]], uncertain shape: [[22363.389]]


In [426]:

# Predict with explanation
mean_pred_exp, std_pred_exp, lime_exp, shap_exp = model.predict_with_explanation(test_encodings[128:129], num_samples=num_samples)

ValueError: Input 0 of layer "dense_553" is incompatible with the layer: expected axis -1 of input shape to have value 1, but received input with shape (5000, 13)

In [381]:
mlp_mean_pred_exp, mlp_std_pred_exp, mlp_lime_exp, mlp_shap_exp = mlp_model.predict_with_exp(test_encodings[128:129], train_encodings, num_samples=num_samples)

ValueError: Found input variables with inconsistent numbers of samples: [5000, 13]

In [372]:
print(mlp_mean_pred_exp)

NameError: name 'mlp_mean_pred_exp' is not defined

In [293]:

print(f"Prediction Mean: {mean_pred_exp}")
print(f"Prediction Std: {std_pred_exp}")
print(f"lime: {len(lime_exp)}")

Prediction Mean: [[464657.6]]
Prediction Std: [[22061.143]]
lime: 10


In [294]:
print(lime_exp)

[<lime.explanation.Explanation object at 0x368db2890>, <lime.explanation.Explanation object at 0x368db2b60>, <lime.explanation.Explanation object at 0x368d7a6b0>, <lime.explanation.Explanation object at 0x368db18a0>, <lime.explanation.Explanation object at 0x368d7b880>, <lime.explanation.Explanation object at 0x368db0550>, <lime.explanation.Explanation object at 0x36559eb00>, <lime.explanation.Explanation object at 0x36559ece0>, <lime.explanation.Explanation object at 0x368d787f0>, <lime.explanation.Explanation object at 0x36559c2b0>]


In [295]:
for i, exp in enumerate(lime_exp):
    print(f"Explanation for instance {i}:")
    for feature, contribution in exp.as_list():
        print(f"  {feature}: {contribution}")
    print("\n")

Explanation for instance 0:
  feature_0 <= -0.00: 0.0
  feature_1 <= 0.00: 0.0
  feature_2 <= -0.00: 0.0
  feature_3 <= 0.00: 0.0
  feature_4 <= 0.00: 0.0
  feature_5 <= 0.00: 0.0
  feature_6 <= 0.00: 0.0
  feature_7 <= -0.00: 0.0
  feature_8 <= -0.00: 0.0
  feature_9 <= 0.00: 0.0
  feature_10 <= 0.00: 0.0
  feature_11 <= -0.00: 0.0
  feature_12 <= -0.00: 0.0


Explanation for instance 1:
  feature_0 <= -0.00: 0.0
  feature_1 <= 0.00: 0.0
  feature_2 <= -0.00: 0.0
  feature_3 <= 0.00: 0.0
  feature_4 <= 0.00: 0.0
  feature_5 <= 0.00: 0.0
  feature_6 <= 0.00: 0.0
  feature_7 <= -0.00: 0.0
  feature_8 <= -0.00: 0.0
  feature_9 <= 0.00: 0.0
  feature_10 <= 0.00: 0.0
  feature_11 <= -0.00: 0.0
  feature_12 <= -0.00: 0.0


Explanation for instance 2:
  feature_0 <= -0.00: 0.0
  feature_1 <= 0.00: 0.0
  feature_2 <= -0.00: 0.0
  feature_3 <= 0.00: 0.0
  feature_4 <= 0.00: 0.0
  feature_5 <= 0.00: 0.0
  feature_6 <= 0.00: 0.0
  feature_7 <= -0.00: 0.0
  feature_8 <= -0.00: 0.0
  feature_9 <= 

In [296]:
print(shap_exp)

[array([[[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]]]), array([[[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]]]), array([[[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]]]), array([[[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]]]), array([[[0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]]]), array([[[0.],
        [0.],
        [0.],
        [0.],
        [0.],
    